In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib as plt

In [ ]:
df = pd.read_csv('smartcart_customers.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

# Data Preprocessing

## 1. Handle the Missing Values

In [ ]:
df['Income'] = df['Income'].fillna(df['Income'].median())

In [ ]:
df.head()

## 2. Feature Engineering

In [ ]:
df.columns

In [ ]:
# Age
df['Age'] = 2026 - df['Year_Birth']

In [ ]:
# Customer Joining Date
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"],dayfirst=True)
ref_date = df["Dt_Customer"].max()
df['Customer_Tenure_days'] = (ref_date - df['Dt_Customer']).dt.days

In [ ]:
# Spending
df['Total_Spending'] = df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] + df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds']

In [ ]:
# Children
df["Total_Children"] = df['Kidhome'] + df['Teenhome']

In [ ]:
# Education

df["Education"] = df["Education"].replace({
        "Basic" : "Undergraduate", "2n Cycle" : "Undergraduate",
        "Graduation" : "Graduate",
        "Master" : "Postgraduate", "PhD" : "Postgraduate"

})

In [ ]:
# Marital Status
df['Living_With'] = df["Marital_Status"].replace({
        "Married" : "Partner", "Together" : "Partner",
        "Single" : "Alone", "Divorced" : "Alone",
        "Widow" : "Alone", "Absurd" : 'Alone', "YOLO" : 'Alone'
})

In [ ]:
df["Living_With"].value_counts()

In [ ]:
df.head()

## 3. Drop Columns

In [ ]:
df.head()

In [ ]:
cols = ["ID","Year_Birth","Marital_Status","Kidhome","Teenhome","Dt_Customer"]
spending_cols = ['MntWines' , 'MntFruits' ,'MntMeatProducts','MntFishProducts', 'MntSweetProducts' , 'MntGoldProds']

cols_to_drop = cols + spending_cols
df_cleaned = df.drop(columns=cols_to_drop)

In [ ]:
df_cleaned.shape

In [ ]:
df.shape

In [ ]:
df_cleaned.head()

## Outliers

In [ ]:
cols = ["Income","Recency","Response","Age","Total_Spending","Total_Children"]

# Relative plots of some features - Pair plot
sns.pairplot(df_cleaned[cols])

In [ ]:
# remove outliers

print("Data size with outliers: ",len(df_cleaned))
df_cleaned = df_cleaned[ (df_cleaned["Age"]<90)]
df_cleaned = df_cleaned[ (df_cleaned["Income"]<600_000)]
print("Data size without outliers: ",len(df_cleaned))

## Heatmap for Correlation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
corr = df_cleaned.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    corr,
    annot=True,
    annot_kws={"size": 6},
    cmap='coolwarm'
)


In [ ]:
df_cleaned.shape

## Encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
ohe = OneHotEncoder()
cat_cols  = ["Education","Living_With"]
enc_cols = ohe.fit_transform(df_cleaned[cat_cols])

In [ ]:
enc_df = pd.DataFrame(enc_cols.toarray(),columns=ohe.get_feature_names_out(cat_cols),index=df_cleaned.index)

In [ ]:
enc_df.head()

In [ ]:
df_encoded = pd.concat([df_cleaned.drop(columns=cat_cols),enc_df],axis = 1)

In [ ]:
df_encoded.shape

In [ ]:
df_encoded.head()

## Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
X = df_encoded

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Visualize

In [ ]:
X_scaled.shape

In [ ]:
# 2D
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

In [ ]:
pca.explained_variance_ratio_

In [ ]:
#  plot
fig = plt.figure(figsize=(8,8))
ax  = fig.add_subplot(111,projection='3d')
ax.scatter(X_pca[:,0],X_pca[:,1],X_pca[:,2])
ax.set_xlabel("PCA1")
ax.set_ylabel("PCA2")
ax.set_zlabel("PCA3")
ax.set_title("3d projection")

# Analyze K value
## 1. Elbow Method

In [ ]:
! pip install kneed

In [ ]:
from sklearn.cluster import KMeans
from kneed import KneeLocator

wcss = []

for k in range(1, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_pca)

    wcss.append(kmeans.inertia_)

In [ ]:
# from sklearn.cluster import KMeans
# from kneed import KneeLocator
# wcss = []
# for k in range(1,11):
#         kmeans = KMeans(n_clusters=k,random_state=42)
#         kmeans.fit_predict(X_pca)
#         wcss.append(kmeans.inertia_)


In [ ]:
knee = KneeLocator(range(1,11),wcss,curve="convex",direction="decreasing")
optimal_k = knee.elbow

In [ ]:
print("Best k = ",optimal_k)

In [ ]:
# plot
plt.plot(range(1,11),wcss,marker = 'o')
plt.xlabel("K")
plt.ylabel("WCSS")

## 2. Silhouette Score

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.metrics import silhouette_score
scores = []
for k in range(2,11):
        kmeans = KMeans(n_clusters=k,random_state=42)
        labels = kmeans.fit_predict(X_pca)
        score = silhouette_score(X_pca,labels)
        scores.append(score)

# Plot
plt.plot(range(2,11),scores,marker = 'o')
plt.xlabel("K")
plt.ylabel("Silhouette Score")

In [ ]:
# Combined Plot
k_range = range(2,11)
fig, ax1 = plt.subplots(figsize=(8,6))
ax1.plot(k_range,wcss[:len(k_range)],marker='o',color='blue')
ax1.set_xlabel("K")
ax1.set_ylabel("WCSS")

ax2 = ax1.twinx()
ax2.plot(k_range,scores[:len(k_range)],marker = 'x',color = 'red',linestyle="--")
ax2.set_ylabel("Silhouette Score")



## Clustering

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "9"

from sklearn.cluster import KMeans

In [ ]:
# K means

kmeans = KMeans(n_clusters=4,random_state=42)
labels_kmeans  = kmeans.fit_predict(X_pca)

In [ ]:
fig = plt.figure(figsize=(8,8))
ax  = fig.add_subplot(111,projection='3d')
ax.scatter(X_pca[:,0],X_pca[:,1],X_pca[:,2],c=labels_kmeans)

In [ ]:
# Agglomerative Clustering
from sklearn.cluster import AgglomerativeClustering

In [ ]:
agg_clf = AgglomerativeClustering(n_clusters=4,linkage='ward')
labels_agg = agg_clf.fit_predict(X_pca)

In [ ]:
fig = plt.figure(figsize=(8,8))
ax  = fig.add_subplot(111,projection='3d')
ax.scatter(X_pca[:,0],X_pca[:,1],X_pca[:,2],c=labels_agg)

## Chareterization of Clusters

In [ ]:
X['Clusters'] = labels_agg

In [ ]:
df_cleaned.head()

In [ ]:
pal = ["red","blue",'yellow',"green"]
sns.countplot(x = X["Clusters"],palette=pal,hue=X["Clusters"])

In [ ]:
# income and Spending patterns

sns.scatterplot(x = X['Total_Spending'],y=X['Income'],hue = X['Clusters'],palette=pal)

In [ ]:
# Cluster Summary

cluster_summary = X.groupby("Clusters").mean()
print(cluster_summary)